# E-Commerce EDA - Week 1 Project
**Dataset:** 500K e-commerce orders, 30 columns

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OrdinalEncoder, OneHotEncoder, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
df = pd.read_csv('ecommerce_500k.csv')
print(df.shape)
df.head()

---
# Phase 1 - Data Audit

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.describe(include='object')

### Missing values

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Count': missing, 'Percent': missing_pct})
missing_df = missing_df[missing_df['Count'] > 0].sort_values('Percent', ascending=False)
missing_df

In [ ]:
missing_df['Percent'].plot(kind='barh', color='salmon', edgecolor='black')
plt.title('Missing Values %')
plt.xlabel('Missing %')
plt.tight_layout()
plt.show()

So `return_reason` has ~88% missing which makes sense because most orders arent returned. The other columns like `shipping_cost`, `days_to_deliver`, `rating`, `customer_age` etc have around 3% missing each - probably just random missing data.

### Checking duplicates

In [ ]:
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate order_ids:", df.duplicated(subset='order_id').sum())

No duplicates found, thats good.

### Data types check

In [ ]:
for col in df.columns:
    print(f"{col}: {df[col].dtype} ({df[col].nunique()} unique)")

`order_date` is stored as object, needs to be converted to datetime. `loyalty_tier` is ordinal (Bronze < Silver < Gold < Platinum) but stored as string.

---
# Phase 2 - Pricing Skewness Analysis

In [ ]:
price_cols = ['product_base_price', 'final_price', 'total_amount', 'shipping_cost']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, col in zip(axes.flatten(), price_cols):
    data = df[col].dropna()
    ax.hist(data, bins=80, color='steelblue', edgecolor='white', alpha=0.8)
    ax.axvline(data.mean(), color='red', linestyle='--', label=f'Mean={data.mean():.1f}')
    ax.axvline(data.median(), color='orange', linestyle='--', label=f'Median={data.median():.1f}')
    ax.set_title(col)
    ax.legend()
plt.suptitle('Price Distributions')
plt.tight_layout()
plt.show()

In [ ]:
for col in price_cols:
    data = df[col].dropna()
    print(f"{col}: skewness={data.skew():.3f}, kurtosis={data.kurtosis():.3f}")

All price columns are positively skewed (right-skewed). `total_amount` is the worst with skewness ~4.1. Mean is always higher than median which confirms the right skew.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 5))
for ax, col in zip(axes, price_cols):
    sns.boxplot(y=df[col].dropna(), ax=ax, color='lightskyblue', flierprops={'marker':'o','markersize':2})
    ax.set_title(col)
plt.suptitle('Box Plots - Price Columns')
plt.tight_layout()
plt.show()

### Trying log transform to fix skewness

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, col in zip(axes.flatten(), price_cols):
    log_data = np.log1p(df[col].dropna())
    ax.hist(log_data, bins=60, color='mediumseagreen', edgecolor='white', alpha=0.8)
    ax.set_title(f'log1p({col}) - skew={log_data.skew():.3f}')
plt.suptitle('After Log Transform')
plt.tight_layout()
plt.show()

Log transform brings the skewness way down and makes the distributions much more normal-looking. Will use log1p in the pipeline.

---
# Phase 3 - Categorical Columns Audit for ML

In [ ]:
cat_cols = df.select_dtypes(include='object').columns.tolist()
for col in cat_cols:
    print(f"{col}: {df[col].nunique()} unique - {df[col].unique()[:5].tolist()}")

In [ ]:
cardinality = df[cat_cols].nunique().sort_values()
cardinality.plot(kind='barh', color='teal', edgecolor='black')
plt.title('Number of Unique Values per Categorical Column')
plt.xlabel('Unique Values')
plt.tight_layout()
plt.show()

Problems I can see:
- `loyalty_tier` has a natural order (Bronze < Silver < Gold < Platinum) so it needs ordinal encoding, not one-hot
- `sub_category` has 45 unique values - thats high cardinality, one-hot will create a lot of columns
- `city` has 15 unique values - manageable with one-hot
- `order_date` is a string, needs to be converted and have features extracted from it
- All the other categoricals need to be encoded before any ML model can use them

---
# Phase 4 - Return Pattern Analysis

In [ ]:
return_rate = df['is_returned'].mean() * 100
print(f"Return rate: {return_rate:.2f}%")
print(f"Returned: {df['is_returned'].sum():,}")
print(f"Not returned: {(df['is_returned']==0).sum():,}")

In [ ]:
returned = df[df['is_returned'] == 1]
reason_counts = returned['return_reason'].value_counts()
reason_counts.plot(kind='bar', color='coral', edgecolor='black')
plt.title('Return Reasons')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
cat_return = df.groupby('category')['is_returned'].mean().sort_values(ascending=False) * 100
cat_return.plot(kind='barh', color='steelblue', edgecolor='black')
plt.title('Return Rate by Category')
plt.xlabel('Return Rate (%)')
plt.tight_layout()
plt.show()

In [ ]:
ship_return = df.groupby('shipping_method')['is_returned'].mean().sort_values(ascending=False) * 100
ship_return.plot(kind='bar', color='teal', edgecolor='black')
plt.title('Return Rate by Shipping Method')
plt.ylabel('Return Rate (%)')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
tier_order = ['Bronze', 'Silver', 'Gold', 'Platinum']
tier_return = df.groupby('loyalty_tier')['is_returned'].mean().reindex(tier_order) * 100
tier_return.plot(kind='bar', color='goldenrod', edgecolor='black')
plt.title('Return Rate by Loyalty Tier')
plt.ylabel('Return Rate (%)')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

disc_bins = pd.cut(df['discount_percentage'], bins=[0,10,20,30,40,50])
disc_return = df.groupby(disc_bins, observed=True)['is_returned'].mean() * 100
disc_return.plot(kind='bar', color='mediumpurple', edgecolor='black', ax=axes[0])
axes[0].set_title('Return Rate by Discount Band')
axes[0].set_ylabel('Return Rate (%)')
axes[0].tick_params(axis='x', rotation=45)

price_bins = pd.qcut(df['final_price'], q=5, duplicates='drop')
price_return = df.groupby(price_bins, observed=True)['is_returned'].mean() * 100
price_return.plot(kind='bar', color='darkorange', edgecolor='black', ax=axes[1])
axes[1].set_title('Return Rate by Price Quintile')
axes[1].set_ylabel('Return Rate (%)')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
first_return = df.groupby('is_first_purchase')['is_returned'].mean() * 100
print("Return rate - Repeat customers:", round(first_return[0], 2), "%")
print("Return rate - First time buyers:", round(first_return[1], 2), "%")

The return rate is about 12% overall. The return reasons are pretty evenly split between Wrong Item, Not as Described, Changed Mind, Better Price Found and Defective.

Looking at the charts, return rate doesnt vary much across categories, shipping methods, or loyalty tiers. Discount level and price also dont seem to have a strong effect. Returns seem to be fairly random which means a prediction model will need good feature engineering to find any signal.

---
# Phase 5 - ColumnTransformer Pipeline

In [ ]:
df['order_date'] = pd.to_datetime(df['order_date'])
df['order_month'] = df['order_date'].dt.month
df['order_dayofweek'] = df['order_date'].dt.dayofweek
df['order_quarter'] = df['order_date'].dt.quarter

target = 'is_returned'
drop_cols = ['order_id', 'customer_id', 'product_id', 'order_date', 'return_reason', 'is_cancelled']

X = df.drop(columns=[target] + drop_cols)
y = df[target]

print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
num_standard = ['final_price', 'total_amount', 'customer_lifetime_value', 'session_duration_mins']
num_minmax = ['discount_percentage', 'quantity', 'rating', 'num_reviews',
              'customer_age', 'pages_viewed', 'clicks_to_purchase',
              'order_month', 'order_dayofweek', 'order_quarter']
num_skewed = ['product_base_price', 'shipping_cost', 'days_to_deliver']
binary_cols = ['is_first_purchase']
cat_onehot = ['category', 'shipping_method', 'payment_method', 'customer_gender', 'country']
cat_high = ['sub_category', 'city']
cat_ordinal = ['loyalty_tier']

In [ ]:
standard_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

minmax_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', MinMaxScaler())
])

skewed_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('log', FunctionTransformer(np.log1p, validate=True)),
    ('scaler', StandardScaler())
])

onehot_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'))
])

ordinal_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(categories=[['Bronze', 'Silver', 'Gold', 'Platinum']]))
])

preprocessor = ColumnTransformer(transformers=[
    ('std', standard_pipe, num_standard),
    ('minmax', minmax_pipe, num_minmax),
    ('skewed', skewed_pipe, num_skewed),
    ('binary', 'passthrough', binary_cols),
    ('onehot', onehot_pipe, cat_onehot),
    ('onehot_h', onehot_pipe, cat_high),
    ('ordinal', ordinal_pipe, cat_ordinal),
], remainder='drop')

print(preprocessor)

In [ ]:
X_transformed = preprocessor.fit_transform(X)
print("Before:", X.shape)
print("After:", X_transformed.shape)
print("Any NaN left:", np.isnan(X_transformed).sum())

In [ ]:
feature_names = preprocessor.get_feature_names_out()
print(f"Total features: {len(feature_names)}")
print("First 15:")
for i, name in enumerate(feature_names[:15]):
    print(f"  {i+1}. {name}")

Pipeline works and transforms everything properly. No NaN values left. The one-hot encoding expanded the feature count quite a bit because of `sub_category` having 45 values.

---
# Phase 6 - Correlation & Feature Importance

In [ ]:
numeric_df = df.select_dtypes(include='number').drop(columns=['order_id', 'customer_id', 'product_id'])
corr_matrix = numeric_df.corr()

fig, ax = plt.subplots(figsize=(14, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, linewidths=0.5, ax=ax, vmin=-1, vmax=1, annot_kws={'size': 8})
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
upper = corr_matrix.where(np.triu(np.ones_like(corr_matrix, dtype=bool))).stack()
high_corr = upper[upper.abs() > 0.7]
high_corr = high_corr[high_corr.index.get_level_values(0) != high_corr.index.get_level_values(1)]
high_corr = high_corr.sort_values(ascending=False)

if len(high_corr) > 0:
    print("Highly correlated pairs (|r| > 0.7):")
    for (c1, c2), val in high_corr.items():
        print(f"  {c1} <-> {c2}: {val:.3f}")
else:
    print("No highly correlated pairs found")

In [ ]:
target_corr = numeric_df.corr()['is_returned'].drop('is_returned').sort_values(key=abs, ascending=False)

colors = ['green' if v > 0 else 'red' for v in target_corr.values]
target_corr.plot(kind='barh', color=colors, edgecolor='black')
plt.title('Correlation with is_returned')
plt.xlabel('Pearson Correlation')
plt.axvline(x=0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

print(target_corr)

In [ ]:
from scipy.stats import chi2_contingency

cat_test_cols = ['category', 'sub_category', 'shipping_method', 'payment_method',
                 'customer_gender', 'country', 'city', 'loyalty_tier']

for col in cat_test_cols:
    ct = pd.crosstab(df[col], df['is_returned'])
    chi2, p, dof, _ = chi2_contingency(ct)
    sig = "YES" if p < 0.05 else "NO"
    print(f"{col}: chi2={chi2:.2f}, p={p:.4f}, significant={sig}")

The price-related columns (base_price, final_price, total_amount) are highly correlated with each other which makes sense. Could drop some of them to avoid multicollinearity.

No numeric feature has a strong correlation with `is_returned` - all the correlations are really weak (below 0.1). `is_cancelled` has the strongest negative correlation.

The chi-squared tests show which categorical variables have a statistically significant relationship with returns.

Overall a prediction model will probably need tree-based methods like Random Forest or XGBoost since linear correlations are so weak.

---
# Done

All 6 phases completed:
1. Data audit - checked missing values, duplicates, data types
2. Pricing skewness - confirmed all price columns are right-skewed, log transform helps
3. Categorical audit - identified ordinal, high-cardinality, and encoding needs
4. Return patterns - ~12% return rate, no single dominant driver
5. ColumnTransformer - full pipeline with different scalers/encoders, zero NaNs after transform
6. Correlation analysis - price columns are inter-correlated, weak individual predictors for returns